In [1]:
# Paramètres
import io
import pandas as pd
import boto3
import pandas as pd
import io
import pandas as pd
import unicodedata
import re
import numpy as np
import unicodedata, re
from pathlib import Path
from pandas.api.types import is_string_dtype
from collections import Counter
from rapidfuzz import process, fuzz
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

In [5]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://192.168.1.230:30137", # service interne K8s
 aws_access_key_id="datalab-team",
 aws_secret_access_key="minio-datalabteam123",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "staging"
object_key = "panel_admin/exports_gold/base_agrege_salaire_brut_2024_2025_avec_zeros.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
# Afficher un aperçu
df.head()

,ANNEE,MOIS_NUM,MOIS,cle_unique,REVENU_TOTAL,NB_POSTES,GRADE_PRINCIPAL,GRADE_SOURCE_PRINCIPAL,SEXE_STD,Code_CITP,Metier_CITP,bareme,bareme_detail,REVENU_TOTAL_W,max_brut,STATUT
0,2024,10,Octobre,884335Q38,175465.0,1,C3,OBSERVE,Homme,0210,Sous-officiers des forces armées,police,police,175465.0,436121.892,En dessous du max
1,2024,1,Janvier,391447S22,305889.0,1,C3,OBSERVE,Femme,2341,Enseignants du primaire,ENS SEC & PRIM,ENS SEC & PRIM_PRIM,305889.0,362501.516,En dessous du max
2,2025,3,Mars,849586Z38,252677.0,1,C3,IMPUTE,Homme,0210,Sous-officiers des forces armées,police,police,252677.0,436121.892,En dessous du max
3,2024,8,Août,318657J22,590781.0,1,B3,OBSERVE,Homme,2341,Enseignants du primaire,ENS SEC & PRIM,ENS SEC & PRIM_PRIM,590781.0,557145.456,Au dessus du max
4,2025,8,Août,297281WFC,685903.0,1,A4,OBSERVE,Femme,2149,Professionnels de l'ingénierie non classés ail...,Tech-Santé,Tech-Santé,685903.0,752453.868,En dessous du max


In [6]:
print("Les colonnes du DataFrame sont :", df.columns.tolist())

Les colonnes du DataFrame sont : ['ANNEE', 'MOIS_NUM', 'MOIS', 'cle_unique', 'REVENU_TOTAL', 'NB_POSTES', 'GRADE_PRINCIPAL', 'GRADE_SOURCE_PRINCIPAL', 'SEXE_STD', 'Code_CITP', 'Metier_CITP', 'bareme', 'bareme_detail', 'REVENU_TOTAL_W', 'max_brut', 'STATUT']


In [ ]:
print("Le nombre d'observations est :", df.shape[0])
print("Le nombre de variables est :", df.shape[1])

Le nombre d'observations est : 7597798
Le nombre de variables est : 14
Répartition des agents par nombre de postes : NB_POSTES
1    7597798
Name: count, dtype: int64


In [18]:
print("Répartition des agents par nombre de postes :")
tab = pd.DataFrame({
    "Effectif": df['NB_POSTES'].value_counts(),
    "Pourcentage": (df['NB_POSTES'].value_counts(normalize=True) * 100).round(2)
})
tab


Répartition des agents par nombre de postes :


,Effectif,Pourcentage
NB_POSTES,,
1,7597798,100.0


In [25]:
df['GRADE_PRINCIPAL'].value_counts()

GRADE_PRINCIPAL
B3    2662647
C3    1674307
A3    1096087
A4     806372
A5     394425
C1     344186
A7     176371
D1     163214
C2     126531
B1      87532
A6      65990
D2        136
Name: count, dtype: int64

In [24]:
# Comparaison égalité / différence
egal = (df["REVENU_TOTAL"] == df["REVENU_TOTAL_W"]).sum()
diff = (df["REVENU_TOTAL"] != df["REVENU_TOTAL_W"]).sum()

# Total des lignes
total = len(df)

# Construction tableau
tab = pd.DataFrame({
    "Effectif": [egal, diff],
    "Proportion (%)": [round(egal/total*100, 2), round(diff/total*100, 2)]
}, index=["Égalité", "Différence"])

tab

,Effectif,Proportion (%)
Égalité,7498929,98.7
Différence,98869,1.3


In [28]:
df.groupby('SEXE_STD')['REVENU_TOTAL_W'].mean()


SEXE_STD
Femme    434496.774057
Homme    507127.327674
Name: REVENU_TOTAL_W, dtype: float64

In [34]:
df.groupby(['ANNEE', 'MOIS_NUM', 'MOIS', 'GRADE_PRINCIPAL'])['REVENU_TOTAL_W'].mean()

ANNEE  MOIS_NUM  MOIS      GRADE_PRINCIPAL
2024   1         Janvier   A3                 5.174195e+05
                           A4                 7.392709e+05
                           A5                 8.333077e+05
                           A6                 1.197485e+06
                           A7                 1.157485e+06
                                                  ...     
2025   12        Décembre  C1                 2.593179e+05
                           C2                 3.209933e+05
                           C3                 2.987492e+05
                           D1                 2.145027e+05
                           D2                 2.354000e+05
Name: REVENU_TOTAL_W, Length: 288, dtype: float64

In [7]:
df['bareme'].unique()

array(['police', 'ENS SEC & PRIM', 'Tech-Santé', 'Barême Général',
       'CORPS PREF', 'ENS SUP',
       'Cadre-Sup-Santé Generaliste & Specialiste', 'MAGISTRAT ',
       'GREFFE', 'CORPS DIPL', None], dtype=object)

In [ ]:
df['bareme'].value_counts()

bareme
ENS SEC & PRIM                               3889928
Barême Général                               1766839
Tech-Santé                                    895320
police                                        669538
Cadre-Sup-Santé Generaliste & Specialiste     157158
ENS SUP                                       135598
GREFFE                                         27514
CORPS PREF                                     21694
MAGISTRAT                                      20961
CORPS DIPL                                     12307
Name: count, dtype: int64

In [43]:
df['bareme'].isna().sum()

np.int64(941)

In [ ]:
df['STATUT'].unique()

array(['En dessous du max', 'Au dessus du max', 'Indéterminé'],
      dtype=object)

: 

In [32]:
# Nombre de NaN
nb_nan = df['REVENU_TOTAL_W'].isna().sum()

# Nombre de valeurs vides (chaînes vides)
nb_vide = (df['REVENU_TOTAL_W'] == "").sum()

# Nombre de valeurs égales à 0
nb_zero = (df['REVENU_TOTAL_W'] == 0).sum()

# Tableau récapitulatif
result = pd.DataFrame({
    "Catégorie": ["NaN", "Vide", "Zéro"],
    "Effectif": [nb_nan, nb_vide, nb_zero]
})

result

,Catégorie,Effectif
0,NaN,0
1,Vide,0
2,Zéro,0


In [ ]:
df['SEXE_STD'].value_counts()